<a href="https://colab.research.google.com/github/JADASIMONESIMON/-Hooplytics-complex-messy-data-project/blob/main/notebooks/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
#import tools
import requests
import pandas as pd
import numpy as np

In [56]:
#API Key
API_KEY = "11f847a9-8965-4f08-8239-98b98e34e2f8"


In [57]:
#calling API Key
url = "https://api.balldontlie.io/v1/players"  # go back to this for now

headers = {
    "Authorization": API_KEY
}

params = {
    "season": 2024,
    "season_type": "regular",
    "type": "base",
    "per_page": 100
}

In [58]:
response = requests.get(url, headers=headers, params=params)

print("Status:", response.status_code)
print("Text preview:", response.text[:300])

data = response.json()
df_2024 = pd.json_normalize(data["data"])
df_2024.head()

Status: 200
Text preview: {"data":[{"id":1,"first_name":"Alex","last_name":"Abrines","position":"G","height":"6-6","weight":"190","jersey_number":"8","college":"FC Barcelona","country":"Spain","draft_year":2013,"draft_round":2,"draft_number":32,"team":{"id":21,"conference":"West","division":"Northwest","city":"Oklahoma City"


,id,first_name,last_name,position,height,weight,jersey_number,college,country,draft_year,draft_round,draft_number,team.id,team.conference,team.division,team.city,team.name,team.full_name,team.abbreviation
0,1,Alex,Abrines,G,6-6,190,8,FC Barcelona,Spain,2013.0,2.0,32.0,21,West,Northwest,Oklahoma City,Thunder,Oklahoma City Thunder,OKC
1,2,Jaylen,Adams,G,6-0,225,10,St. Bonaventure,USA,NaN,NaN,NaN,1,East,Southeast,Atlanta,Hawks,Atlanta Hawks,ATL
2,3,Steven,Adams,C,6-11,265,12,Pittsburgh,New Zealand,2013.0,1.0,12.0,11,West,Southwest,Houston,Rockets,Houston Rockets,HOU
3,4,Bam,Adebayo,C-F,6-9,255,13,Kentucky,USA,2017.0,1.0,14.0,16,East,Southeast,Miami,Heat,Miami Heat,MIA
4,5,DeVaughn,Akoon-Purcell,G-F,6-5,201,44,Illinois State,Trinidad and Tobago,2016.0,NaN,NaN,8,West,Northwest,Denver,Nuggets,Denver Nuggets,DEN


In [59]:
# TASK 1: Clean data (Avnoor)

# - Handle missing values
df_2024 = df_2024.fillna(0)

# - Convert columns to numeric
numeric_cols = ["height_feet", "height_inches", "weight_pounds"]
for col in numeric_cols:
    if col in df_2024.columns:
        df_2024[col] = pd.to_numeric(df_2024[col], errors='coerce')

# - Rename columns
df_2024 = df_2024.rename(columns={
    "first_name": "FirstName",
    "last_name": "LastName",
    "height_feet": "HeightFeet",
    "height_inches": "HeightInches",
    "weight_pounds": "Weight"
})

# - Remove bad rows (duplicates)
df_2024 = df_2024.drop_duplicates()

# Final output
df_2024.head()


,id,FirstName,LastName,position,height,weight,jersey_number,college,country,draft_year,draft_round,draft_number,team.id,team.conference,team.division,team.city,team.name,team.full_name,team.abbreviation
0,1,Alex,Abrines,G,6-6,190,8,FC Barcelona,Spain,2013.0,2.0,32.0,21,West,Northwest,Oklahoma City,Thunder,Oklahoma City Thunder,OKC
1,2,Jaylen,Adams,G,6-0,225,10,St. Bonaventure,USA,0.0,0.0,0.0,1,East,Southeast,Atlanta,Hawks,Atlanta Hawks,ATL
2,3,Steven,Adams,C,6-11,265,12,Pittsburgh,New Zealand,2013.0,1.0,12.0,11,West,Southwest,Houston,Rockets,Houston Rockets,HOU
3,4,Bam,Adebayo,C-F,6-9,255,13,Kentucky,USA,2017.0,1.0,14.0,16,East,Southeast,Miami,Heat,Miami Heat,MIA
4,5,DeVaughn,Akoon-Purcell,G-F,6-5,201,44,Illinois State,Trinidad and Tobago,2016.0,0.0,0.0,8,West,Northwest,Denver,Nuggets,Denver Nuggets,DEN


In [66]:
# TASK 2: Feature Engineering



# Load cleaned data
df = pd.read_csv("nba_2024_cleaned.csv")

# Create league totals from player data
totals = df.sum()

# Build single-row dataset for the season
df_league = pd.DataFrame({
    'Season': [2024],
    'PTS': [totals['PTS']],
    'FG': [totals['FG']],
    'FGA': [totals['FGA']],
    '3P': [totals['3P']],
    '3PA': [totals['3PA']]
})

# - Create total shots column
df_league['TotalShots'] = df_league['FGA']

# - Calculate % of 3PT shots
df_league['ThreePtAttemptShare'] = (df_league['3PA'] / df_league['FGA']) * 100
df_league['ThreePtPercent'] = (df_league['3P'] / df_league['3PA']) * 100

# - Calculate % of 2PT shots
df_league['TwoPtAttempts'] = df_league['FGA'] - df_league['3PA']
df_league['TwoPtMade'] = df_league['FG'] - df_league['3P']
df_league['TwoPtPercent'] = (df_league['TwoPtMade'] / df_league['TwoPtAttempts']) * 100

# Clean output
df_league = df_league.round(2)

print(df_league)

   Season       PTS        FG       FGA       3P      3PA  TotalShots  \
0    2024  310834.0  114772.0  243062.0  35290.0  96891.0    243062.0   

   ThreePtAttemptShare  ThreePtPercent  TwoPtAttempts  TwoPtMade  TwoPtPercent  
0                39.86           36.42       146171.0    79482.0         54.38  


In [ ]:
# TASK 3: Analysis (Bennett)
seasons = ['2024', '2025', '2026']
all_total = []

for year in seasons:
  url = f"https://www.basketball-reference.com/leagues/NBA_{year}_totals.html"
  tables = pd.read_html(url)

  for i, table in enumerate(tables):
      if 'PTS' in table.columns and len(table) > 20:
          df = table
          break
  else:
      print(f"No table found for {year}")
      continue

  cols = [col for col in ['PTS', '3P','3PA','FG','FGA'] if col in df.columns]
  totals = df[cols].sum(numeric_only=True).round(0)
  totals['Season'] = year
  all_total.append(totals)

# - Sum league totals
df_league = pd.DataFrame(all_total)
pts = df_league['PTS'].sum()
three_pt = df_league.get('3P', pd.Series([0])).sum()
three_pa = df_league.get('3PA', pd.Series([0])).sum()
fg = df_league.get('FG', pd.Series([0])).sum()
fga = df_league.get('FGA', pd.Series([0])).sum()

print(f"\nLeague Totals (2024-2026)")
print(f"Total Points: {pts:,.0f}")
print(f"Three Pointers Made / Attempted: {three_pt:,.0f}/{three_pa:,.0f}")
print(f"Field Goals Made / Attempted: {fg:,.0f}/{fga:,.0f}")

# - Calculate overall percentages
if three_pa > 0:
    three_pct = (three_pt / three_pa * 100).round(1)
else:
    three_pct = 0
if fga > 0:
    fg_pct = (fg / fga * 100).round(1)
else:
    fg_pct = 0

print(f"\nOverall Percentages")
print(f"League Three Pointers: {three_pct}%")
print(f"League Field Goals: {fg_pct}%")

# - Compare 3PT vs 2PT
print(f"\n3PT vs 2PT")
print(f"3PT are {three_pa / fga * 100:,.1f}% of FGA")

# - Loop through multiple seasons
print(f"\nSeasons")
print(df_league[['Season','PTS']].round(0))


League Totals (2024-2026)
Total Points: 907,675
Three Pointers Made / Attempted: 105,266/291,641
Field Goals Made / Attempted: 332,162/708,082

Overall Percentages
League Three Pointers: 36.1%
League Field Goals: 46.9%

3PT vs 2PT
3PT are 41.2% of FGA

Seasons
  Season       PTS
0   2024  310834.0
1   2025  311448.0
2   2026  285393.0


In [ ]:
#TASK 4 ; visualiztion + presentation
# - Bar chart (3PT vs 2PT)
# - Trend line over time
# - Clean notebook layout
# - Write explanation (IMPORTANT for grade)




In [ ]:
# TASK 4: Visualization + Presentation

import matplotlib.pyplot as plt

# --- Rebuild season-by-season totals cleanly ---
seasons = ['2024', '2025', '2026']
all_total = []

for year in seasons:
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_totals.html"
    tables = pd.read_html(url)

    df = None
    for table in tables:
        if 'PTS' in table.columns and len(table) > 20:
            df = table.copy()
            break

    if df is None:
        print(f"No table found for {year}")
        continue

    # Keep only needed numeric columns
    cols = [col for col in ['PTS', '3P', '3PA', 'FG', 'FGA'] if col in df.columns]
    season_totals = df[cols].sum(numeric_only=True)
    season_totals['Season'] = int(year)
    all_total.append(season_totals)

# Create season summary dataframe
df_league = pd.DataFrame(all_total)

# Feature engineering for visualization
df_league['TwoPA'] = df_league['FGA'] - df_league['3PA']
df_league['ThreePtShare'] = (df_league['3PA'] / df_league['FGA']) * 100
df_league['TwoPtShare'] = (df_league['TwoPA'] / df_league['FGA']) * 100
df_league['FGPercent'] = (df_league['FG'] / df_league['FGA']) * 100
df_league['ThreePtPercent'] = (df_league['3P'] / df_league['3PA']) * 100

df_league = df_league.round(2)

print("Season summary:")
print(df_league[['Season', 'PTS', 'FGA', '3PA', 'TwoPA', 'ThreePtShare', 'TwoPtShare']])

# --- Chart 1: Bar chart for overall 3PT vs 2PT attempts ---
total_3pa = df_league['3PA'].sum()
total_2pa = df_league['TwoPA'].sum()

plt.figure(figsize=(8,5))
plt.bar(['3-Point Attempts', '2-Point Attempts'], [total_3pa, total_2pa])
plt.title('NBA Shot Attempts: 3PT vs 2PT (2024–2026)')
plt.ylabel('Total Attempts')
plt.xlabel('Shot Type')
plt.tight_layout()
plt.show()

# --- Chart 2: Trend line over time for 3PT shot share ---
plt.figure(figsize=(8,5))
plt.plot(df_league['Season'], df_league['ThreePtShare'], marker='o')
plt.title('Trend in 3-Point Attempt Share Over Time')
plt.xlabel('Season')
plt.ylabel('3PT Attempts as % of All FGA')
plt.xticks(df_league['Season'])
plt.tight_layout()
plt.show()

# --- Optional Chart 3: Points by season ---
plt.figure(figsize=(8,5))
plt.plot(df_league['Season'], df_league['PTS'], marker='o')
plt.title('Total League Points by Season')
plt.xlabel('Season')
plt.ylabel('Points')
plt.xticks(df_league['Season'])
plt.tight_layout()
plt.show()

Explanation (Alyssa)
This visualization compares how NBA teams attempt 3-point shots versus 2-point shots across the 2024–2026 seasons. The bar chart shows the total volume of each shot type, while the trend line shows how the share of 3-point attempts changes over time.

The results suggest that 3-point shooting makes up a large portion of total field goal attempts, which reflects how modern NBA offenses rely more heavily on spacing and perimeter shooting. Even though 2-point attempts still account for a larger total share, the trend line helps show whether teams are continuing to increase their dependence on the three-point shot each season.

These visuals support the project goal by turning cleaned basketball data into an easy-to-understand summary of league-wide scoring behavior.